# Decode deconvolved calcium with `SortedSpikesDecoder`

This notebook mirrors the sorted-spikes tutorial workflow, but replaces ground-truth spike trains with OASIS-deconvolved calcium spikes.

It covers three cases:

1. **Baseline decode**: fit on run A, decode run B, and decode a continuous replay event.
2. **Noise effect**: visualize decoding at `sigma = 1.0, 4.0, 8.0`, then sweep sigma values and measure median absolute decoding error.
3. **Speed effect**: compare run decoding at three running speeds.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from replay_trajectory_classification.calcium_sorted_spikes_decoding import (
    deconvolve_and_binarize,
    make_sorted_spikes_decoder,
    maximum_a_posteriori_position,
    median_decoding_error,
)
from replay_trajectory_classification.simulate import simulate_position, simulate_time
from replay_trajectory_classification.simulate_calcium import (
    make_continuous_replay,
    make_simulated_run_data,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_context("talk", font_scale=0.8)
np.random.seed(0)

In [ ]:
TRACK_HEIGHT = 120.0
N_NEURONS = 12
PLACE_FIELD_MEANS = np.linspace(0.0, TRACK_HEIGHT, N_NEURONS)
RUN_SAMPLING_FREQUENCY = 30
REPLAY_SAMPLING_FREQUENCY = 200
INTERNAL_SAMPLING_FREQUENCY = 1000
N_RUNS = 2
BASE_SIGMA = 1.0
BASE_RUNNING_SPEED = 10.0
REPLAY_SPEEDUP = 120
POSITION_STD = 3.0
FIXED_SIGMAS = [1.0, 4.0, 8.0]
SIGMA_SWEEP = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0])
SPEED_VALUES = [5.0, 15.0, 30.0]

In [ ]:
def make_run_dataset(rng_seed, sigma=BASE_SIGMA, running_speed=BASE_RUNNING_SPEED):
    time, position, sampling_frequency, calcium_traces, true_spikes, place_fields = make_simulated_run_data(
        sampling_frequency=RUN_SAMPLING_FREQUENCY,
        track_height=TRACK_HEIGHT,
        running_speed=running_speed,
        n_runs=N_RUNS,
        place_field_means=PLACE_FIELD_MEANS,
        sigma=sigma,
        rng=np.random.default_rng(rng_seed),
    )
    inferred_spikes = deconvolve_and_binarize(calcium_traces)
    return {
        "time": time,
        "position": position,
        "sampling_frequency": sampling_frequency,
        "calcium_traces": calcium_traces,
        "true_spikes": true_spikes,
        "inferred_spikes": inferred_spikes,
        "place_fields": place_fields,
        "sigma": sigma,
        "running_speed": running_speed,
    }


def make_true_continuous_replay_position(
    sampling_frequency=REPLAY_SAMPLING_FREQUENCY,
    internal_sampling_frequency=INTERNAL_SAMPLING_FREQUENCY,
    track_height=TRACK_HEIGHT,
    running_speed=BASE_RUNNING_SPEED,
    replay_speedup=REPLAY_SPEEDUP,
    is_outbound=True,
):
    replay_speed = running_speed * replay_speedup
    n_samples = int(np.ceil(2 * internal_sampling_frequency * track_height / replay_speed))
    replay_time_internal = simulate_time(n_samples, internal_sampling_frequency)
    replay_position_internal = simulate_position(replay_time_internal, track_height, replay_speed)
    half_n_samples = n_samples // 2
    if is_outbound:
        replay_position_internal = replay_position_internal[:half_n_samples]
    else:
        replay_position_internal = replay_position_internal[-half_n_samples:]
    subsample_factor = internal_sampling_frequency // sampling_frequency
    return replay_position_internal[::subsample_factor]


def make_replay_dataset(rng_seed, sigma=BASE_SIGMA, running_speed=BASE_RUNNING_SPEED):
    time, true_spikes, calcium_traces = make_continuous_replay(
        sampling_frequency=REPLAY_SAMPLING_FREQUENCY,
        internal_sampling_frequency=INTERNAL_SAMPLING_FREQUENCY,
        track_height=TRACK_HEIGHT,
        running_speed=running_speed,
        place_field_means=PLACE_FIELD_MEANS,
        replay_speedup=REPLAY_SPEEDUP,
        sigma=sigma,
        rng=np.random.default_rng(rng_seed),
    )
    true_position = make_true_continuous_replay_position(running_speed=running_speed)[: len(time)]
    inferred_spikes = deconvolve_and_binarize(calcium_traces)
    return {
        "time": time,
        "position": true_position,
        "sampling_frequency": REPLAY_SAMPLING_FREQUENCY,
        "calcium_traces": calcium_traces,
        "true_spikes": true_spikes,
        "inferred_spikes": inferred_spikes,
        "sigma": sigma,
        "running_speed": running_speed,
    }


def fit_decoder(run_dataset):
    decoder = make_sorted_spikes_decoder(
        position=run_dataset["position"],
        sampling_frequency=run_dataset["sampling_frequency"],
        position_std=POSITION_STD,
    )
    decoder.fit(run_dataset["position"], run_dataset["inferred_spikes"])
    return decoder


def decode_dataset(decoder, dataset):
    return decoder.predict(dataset["inferred_spikes"], time=dataset["time"])


def raster_points(spikes):
    spike_time_ind, neuron_ind = np.nonzero(spikes)
    return spike_time_ind, neuron_ind


def plot_baseline_decode(dataset, results, title):
    fig, axes = plt.subplots(3, 1, sharex=True, constrained_layout=True, figsize=(14, 7))
    spike_time_ind, neuron_ind = raster_points(dataset["inferred_spikes"])
    cmap = plt.get_cmap("tab20")
    colors = [cmap.colors[ind % len(cmap.colors)] for ind in neuron_ind]

    axes[0].scatter(dataset["time"][spike_time_ind], neuron_ind + 1, c=colors, s=5, clip_on=False)
    axes[0].set_ylabel("Cells")
    axes[0].set_yticks((1, dataset["inferred_spikes"].shape[1]))
    axes[0].set_title(title)

    results.causal_posterior.plot(
        x="time", y="position", ax=axes[1], cmap="bone_r", vmin=0.0, vmax=0.08, add_colorbar=False
    )
    axes[1].plot(dataset["time"], dataset["position"], color="magenta", linestyle="--", linewidth=2)
    axes[1].set_ylabel("Position")
    axes[1].set_xlabel("")
    axes[1].set_title("Causal posterior")

    results.acausal_posterior.plot(
        x="time", y="position", ax=axes[2], cmap="bone_r", vmin=0.0, vmax=0.08, add_colorbar=False
    )
    axes[2].plot(dataset["time"], dataset["position"], color="magenta", linestyle="--", linewidth=2)
    axes[2].set_ylabel("Position")
    axes[2].set_xlabel("Time [s]")
    axes[2].set_title("Acausal posterior")
    sns.despine(offset=5)
    return fig, axes


def plot_overlay_panel(ax, dataset, results, title, posterior_name="acausal_posterior"):
    posterior = getattr(results, posterior_name)
    posterior.plot(x="time", y="position", ax=ax, cmap="bone_r", vmin=0.0, vmax=0.08, add_colorbar=False)
    ax.plot(dataset["time"], dataset["position"], color="magenta", linestyle="--", linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Position")


def summarize_errors(rows, sort_key):
    return pd.DataFrame(rows).sort_values(sort_key).reset_index(drop=True)

## 1. Baseline decode: fit on run A, decode run B, decode continuous replay

In [ ]:
run_a = make_run_dataset(rng_seed=0, sigma=BASE_SIGMA, running_speed=BASE_RUNNING_SPEED)
run_b = make_run_dataset(rng_seed=1, sigma=BASE_SIGMA, running_speed=BASE_RUNNING_SPEED)
replay_b = make_replay_dataset(rng_seed=2, sigma=BASE_SIGMA, running_speed=BASE_RUNNING_SPEED)

decoder = fit_decoder(run_a)
run_b_results = decode_dataset(decoder, run_b)
replay_b_results = decode_dataset(decoder, replay_b)

run_b_error = median_decoding_error(run_b_results.acausal_posterior, run_b["position"])
replay_b_map = maximum_a_posteriori_position(replay_b_results.acausal_posterior)

print(f"Run B median absolute error: {run_b_error:.2f} cm")
print(f"Replay B decoded position range: {replay_b_map.min():.2f} to {replay_b_map.max():.2f} cm")
run_b_results

In [ ]:
plot_baseline_decode(
    run_b,
    run_b_results,
    title="Run B decode from deconvolved calcium spikes",
);

In [ ]:
plot_baseline_decode(
    replay_b,
    replay_b_results,
    title="Continuous replay decode from deconvolved calcium spikes",
);

## 2. Noise effect

In [ ]:
noise_visual_results = []
for sigma in FIXED_SIGMAS:
    train_dataset = make_run_dataset(rng_seed=100 + int(sigma * 10), sigma=sigma, running_speed=BASE_RUNNING_SPEED)
    test_dataset = make_run_dataset(rng_seed=200 + int(sigma * 10), sigma=sigma, running_speed=BASE_RUNNING_SPEED)
    decoder = fit_decoder(train_dataset)
    results = decode_dataset(decoder, test_dataset)
    error = median_decoding_error(results.acausal_posterior, test_dataset["position"])
    noise_visual_results.append((sigma, test_dataset, results, error))

fig, axes = plt.subplots(1, len(noise_visual_results), figsize=(16, 4), sharey=True, constrained_layout=True)
for ax, (sigma, dataset, results, error) in zip(axes, noise_visual_results):
    plot_overlay_panel(ax, dataset, results, title=f"sigma={sigma:.1f}\nmedian error={error:.2f} cm")
axes[0].set_ylabel("Position [cm]")
sns.despine(offset=5)

In [ ]:
noise_error_rows = []
for sigma in SIGMA_SWEEP:
    train_dataset = make_run_dataset(rng_seed=1000 + int(sigma * 100), sigma=float(sigma), running_speed=BASE_RUNNING_SPEED)
    test_dataset = make_run_dataset(rng_seed=2000 + int(sigma * 100), sigma=float(sigma), running_speed=BASE_RUNNING_SPEED)
    decoder = fit_decoder(train_dataset)
    results = decode_dataset(decoder, test_dataset)
    noise_error_rows.append({
        "sigma": float(sigma),
        "median_abs_error": median_decoding_error(results.acausal_posterior, test_dataset["position"]),
    })

noise_error_df = summarize_errors(noise_error_rows, "sigma")
noise_error_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(noise_error_df["sigma"], noise_error_df["median_abs_error"], marker="o")
plt.xlabel("Calcium noise sigma")
plt.ylabel("Median absolute error [cm]")
plt.title("Decoding error vs. calcium noise")
sns.despine(offset=5)

## 3. Speed effect

In [ ]:
speed_results = []
for running_speed in SPEED_VALUES:
    train_dataset = make_run_dataset(rng_seed=300 + int(running_speed), sigma=BASE_SIGMA, running_speed=running_speed)
    test_dataset = make_run_dataset(rng_seed=400 + int(running_speed), sigma=BASE_SIGMA, running_speed=running_speed)
    decoder = fit_decoder(train_dataset)
    results = decode_dataset(decoder, test_dataset)
    error = median_decoding_error(results.acausal_posterior, test_dataset["position"])
    speed_results.append((running_speed, test_dataset, results, error))

fig, axes = plt.subplots(1, len(speed_results), figsize=(16, 4), sharey=True, constrained_layout=True)
for ax, (running_speed, dataset, results, error) in zip(axes, speed_results):
    plot_overlay_panel(ax, dataset, results, title=f"speed={running_speed:.0f} cm/s\nmedian error={error:.2f} cm")
axes[0].set_ylabel("Position [cm]")
sns.despine(offset=5)

In [ ]:
speed_error_df = summarize_errors(
    [
        {"running_speed": running_speed, "median_abs_error": error}
        for running_speed, _dataset, _results, error in speed_results
    ],
    "running_speed",
)
speed_error_df